# Clase 7 · Notebook 2: las funciones ya listas, con sus problemas dichos en voz alta

**El problema.** En el Notebook 1 construimos la lógica paso a paso. Pero un pipeline no corre desde un notebook: corre desde archivos `.py` que un DAG, un `make` o un job de Glue pueden llamar. Falta saber **cuáles funciones van en cuáles archivos**, con qué firma exacta, y qué hacen bien y qué hacen mal a propósito.

**Qué hay aquí.** Las mismas funciones, ya escritas, con la **misma firma** que tienen en los `.py`. Cada celda de código empieza diciendo a qué archivo y a qué función se copia. Importamos las herramientas de la casa (nunca las copiamos) y definimos solo las funciones del pipeline.

**Dos problemas puestos a propósito, que decimos en voz alta:**

1. **Bronze no agrega las columnas de metadatos.** Sin ellas, nada dice de qué lote ni de qué archivo vino cada fila, y no hay huella de la fila.
2. **Silver no deduplica.** Si un mes entra dos veces a bronze, silver lo guarda dos veces, y gold lo multiplica.

Y un tercer aviso: **gold responde una pregunta de la casa** (¿a qué hora del día y en qué barrio se paga más por milla?). No es la pregunta del Taller 2: la tuya es otra.

| Celda | Función | Va en | Qué pasa con ella |
|---|---|---|---|
| 3 | `agregar_metadatos` | `src/pipeline/bronze.py` | **Problema 1**: no agrega nada. La escribe tu equipo. |
| 5 | `debe_ingestar` y `registrar_ingesta` | `src/pipeline/bronze.py` | Listas: la puerta y el registro de la tabla de control |
| 6 | `deduplicar` | `src/pipeline/silver.py` | **Problema 2**: devuelve la tabla tal cual. La escribe tu equipo. |
| 8 | `tarifa_por_milla_por_barrio` (más su franja del día y sus listas) | `src/pipeline/gold.py` | Lista: ejemplo de la casa, no es el Taller |
| 10 | `DDL_TARIFA_POR_MILLA` y `TABLAS_A_CARGAR` | `src/pipeline/carga.py` | Lista: el `CREATE TABLE` de gold en Snowflake |
| 12 | Comparación con los `.py` | | Confirma que son las mismas que trae el repositorio |
| 13 | `probar_mis_funciones` | | Las mismas pruebas de `make test` |

> **Antes de empezar.** En esta máquina solo cabe un proceso de Spark a la vez. Si Airflow del Code Editor está encendido, apágalo con `cd ~/bigdata && make airflow-local-down`, y si dejaste abierto el Notebook 1, ciérralo o reinicia su kernel. Este notebook no lee ni escribe en S3: solo prueba funciones con filas escritas a mano.

## 0. Preparación

El kernel debe ser **Python (bigdata)**. La celda deja el repositorio en la ruta de búsqueda para poder escribir `from pipeline... import ...`.

In [ ]:
# Celda 1
import os, shutil, sys
from pathlib import Path

# Java: el kernel del notebook no lee ~/.bashrc, así que puede no ver el Java que instaló make setup.
if shutil.which("java") is None:
    for candidato in [os.environ.get("JAVA_HOME"), str(Path.home() / ".local/opt/java17")]:
        if candidato and (Path(candidato) / "bin/java").exists():
            os.environ["JAVA_HOME"] = candidato
            os.environ["PATH"] = f"{candidato}/bin:" + os.environ["PATH"]
            break

print("Python:", sys.version.split()[0])
print("Ejecutable:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este kernel no es el del proyecto. Elige «Python (bigdata)» arriba a la derecha y reinicia el notebook.")

# La raíz del repositorio es la carpeta que contiene pyproject.toml.
RAIZ = Path.home() / "bigdata"
for carpeta in [Path.cwd(), *Path.cwd().parents]:
    if (carpeta / "pyproject.toml").exists():
        RAIZ = carpeta
        break
sys.path.insert(0, str(RAIZ / "src"))   # así se puede hacer «from pipeline... import ...»
print("Repositorio:", RAIZ)

Spark con la misma configuración del pipeline (`code_editor`: `local[2]`, 2 GB). Aquí no se toca S3, así que no hacen falta los `.jar` de S3A.

In [ ]:
# Celda 2
from pipeline.engine import get_spark

spark = get_spark("code_editor")
print("Versión de Spark:", spark.version)

## 1. Bronze: `agregar_metadatos` (problema 1)

**Qué hace la función cuando está bien escrita.** Recibe el DataFrame que salió del crudo y le agrega cinco columnas, después de las originales y en este orden: `_fecha_ingesta`, `_id_lote`, `_hash_fila`, `_fuente` y `_archivo_origen` (la lista está en `COLUMNAS_METADATOS`). La huella `_hash_fila` identifica cada fila por sus valores originales: dos filas iguales dan la misma huella, y no depende del lote.

**Qué hace hoy.** **Nada: devuelve `df` tal cual.** Lo decimos en voz alta: este es el primer problema a propósito. Bronze guarda las filas sin decir de qué lote ni de qué archivo vienen, y sin la huella silver no podrá saber qué filas ya vio.

> **Copia a:** `src/pipeline/bronze.py`, función `agregar_metadatos`. La marca `# PENDIENTE(equipo)` dentro del cuerpo es lo que buscan las pruebas para decir `PENDIENTE` en lugar de `FALLA`. La borra quien escriba la función de verdad.

In [ ]:
# Celda 3
# → Copia a src/pipeline/bronze.py · función agregar_metadatos
from datetime import datetime

from pyspark.sql import DataFrame

from pipeline.herramientas import Corrida


def agregar_metadatos(
    df: DataFrame, fuente: str, archivo_origen: str, id_lote: str, fecha_ingesta: datetime
) -> DataFrame:
    # PENDIENTE(equipo): hoy esta función devuelve df tal cual, sin metadatos. Por eso bronze
    # no guarda de qué lote ni de qué archivo vino cada fila, y silver no puede deduplicar.
    return df

Veamos el problema con dos filas escritas a mano. La función debería dejar las dos columnas originales más las cinco de metadatos.

In [ ]:
# Celda 4
from datetime import UTC

from pipeline.herramientas import COLUMNAS_METADATOS

mini = spark.createDataFrame([("a", 1), ("b", 2)], "letra string, numero int")
resultado = agregar_metadatos(mini, "https://fuente/x", "x.parquet", "lote1", datetime.now(UTC))

print("Columnas que debería tener:", ["letra", "numero", *COLUMNAS_METADATOS])
print("Columnas que tiene        :", resultado.columns)
print("Faltan                    :", [c for c in COLUMNAS_METADATOS if c not in resultado.columns])

## 2. Bronze: `debe_ingestar` y `registrar_ingesta` (listas)

Son las dos funciones que construimos en el Notebook 1. Aquí están como las tiene `bronze.py`. Usan dos herramientas de la casa: `mes_ya_procesado` (la puerta de la tabla de control, con la regla de las tres filas) y `registrar_control` (escribe una fila en la tabla).

- `debe_ingestar(corrida, tabla, mes)` devuelve `True` si hay que ingestar y `False` si hay que saltarse el mes. Con registro y `reprocesar=false` salta; con `reprocesar=true` o sin registro, ingesta.
- `registrar_ingesta(corrida, tabla, mes, fuente, leidas, filas, inicio)` anota que bronze terminó bien. **Se llama solo después de escribir el lote**: si la escritura falla, no queda registro y el mes se puede reintentar.

> **Copia a:** `src/pipeline/bronze.py`, funciones `debe_ingestar` y `registrar_ingesta`. Ninguna tiene problema.

In [ ]:
# Celda 5
# → Copia a src/pipeline/bronze.py · funciones debe_ingestar y registrar_ingesta
from pipeline.herramientas import mes_ya_procesado, registrar_control

CAPA = "bronze"


def debe_ingestar(corrida: Corrida, tabla: str, mes: str) -> bool:
    ya_esta_registrado = mes_ya_procesado(corrida, tabla, CAPA, mes)
    if ya_esta_registrado:
        return False
    return True


def registrar_ingesta(
    corrida: Corrida,
    tabla: str,
    mes: str,
    fuente: str,
    leidas: int,
    filas: int,
    inicio: datetime,
) -> None:
    registrar_control(corrida, tabla, CAPA, mes, filas, inicio, fuente, leidas)

## 3. Silver: `deduplicar` (problema 2)

**Qué hace la función cuando está bien escrita.** Deja **una** fila por cada fila distinta. Silver lee **todos** los lotes de bronze del mes; si el mes entró dos veces (porque se reprocesó), cada fila llega dos veces. `deduplicar` es lo que hace de silver el espejo limpio de la fuente: pase lo que pase en bronze, silver queda igual. Necesita que bronze haya agregado la huella `_hash_fila`.

**Qué hace hoy.** **Nada: devuelve `df` tal cual.** Lo decimos en voz alta: este es el segundo problema a propósito. Al reprocesar un mes, silver tiene cada fila dos veces, y gold, que une viajes, zonas y clima, las multiplica otra vez: con dos lotes, los viajes de gold salen ×8.

> **Copia a:** `src/pipeline/silver.py`, función `deduplicar`. Igual que la anterior, la marca `# PENDIENTE(equipo)` es lo que buscan las pruebas.

In [ ]:
# Celda 6
# → Copia a src/pipeline/silver.py · función deduplicar
def deduplicar(df: DataFrame) -> DataFrame:
    # PENDIENTE(equipo): hoy esta función devuelve df tal cual. Por eso, al reprocesar un mes,
    # silver tiene cada fila dos veces (una por lote de bronze) y gold las multiplica.
    return df

Lo vemos con cuatro filas que son dos filas repetidas. La huella la calcula `hash_fila`, la herramienta de la casa: dos filas con los mismos valores dan la misma huella.

In [ ]:
# Celda 7
from pyspark.sql import functions as F

from pipeline.herramientas import hash_fila

filas = [("a", 1), ("b", 2)]
dos_veces = spark.createDataFrame(filas + filas, "letra string, numero int")
dos_veces = dos_veces.withColumn("_hash_fila", hash_fila(["letra", "numero"]))

print("Filas de entrada       :", dos_veces.count())
print("Huellas distintas      :", dos_veces.select("_hash_fila").distinct().count())
print("Filas después de deduplicar:", deduplicar(dos_veces).count(), "(deberían ser 2)")

## 4. Gold: `tarifa_por_milla_por_barrio` (una pregunta de la casa)

**La pregunta.** «¿A qué hora del día y en qué barrio se paga más por milla?» Gold la responde con una tabla pequeña: por fecha, barrio y franja del día (madrugada, mañana, tarde, noche), cuántos viajes hubo, cuántas millas sumaron, cuánto se pagó y cuánto sale **por milla**. La tarifa por milla es el cociente de las **sumas**, no el promedio de las tarifas.

**No es la pregunta del Taller.** Cada equipo escribe la suya. Esta es solo la receta: una función que recibe el enriquecido (viajes con barrio, zona y clima) y devuelve la tabla de gold. Para agregar la tuya, escribes la función y la sumas a `TABLAS_GOLD` (y a `LLAVES_DE_GOLD`, que dice qué columnas identifican una fila; gold exige que no se repitan ni sean nulas). Gold no tiene huecos.

> **Copia a:** `src/pipeline/gold.py`, todo el bloque de esta celda (las constantes, `_franja_del_dia`, `tarifa_por_milla_por_barrio`, `TABLAS_GOLD` y `LLAVES_DE_GOLD`).

In [ ]:
# Celda 8
# → Copia a src/pipeline/gold.py · función tarifa_por_milla_por_barrio y sus listas
from pyspark.sql import functions as F

COLUMNAS_TARIFA_POR_MILLA = [
    "fecha",
    "borough",
    "franja",
    "viajes",
    "distancia_total_mi",
    "ingreso_total",
    "tarifa_por_milla",
]

# Para los viajes cuya zona de recogida no está en el lookup (el barrio queda nulo).
BARRIO_DESCONOCIDO = "Desconocido"


def _franja_del_dia(hora: F.Column) -> F.Column:
    """madrugada (0 a 5 h), mañana (6 a 11), tarde (12 a 17) o noche (18 a 23)."""
    return (
        F.when(hora < 6, "madrugada")
        .when(hora < 12, "mañana")
        .when(hora < 18, "tarde")
        .otherwise("noche")
    )


def tarifa_por_milla_por_barrio(enriquecido: DataFrame) -> DataFrame:
    """Por fecha, barrio y franja del día: viajes, millas, ingreso y cuánto se paga por milla."""
    con_franja = enriquecido.select(
        F.col("pickup_date").alias("fecha"),
        F.coalesce(F.col("borough"), F.lit(BARRIO_DESCONOCIDO)).alias("borough"),
        _franja_del_dia(F.hour("pickup_datetime")).alias("franja"),
        F.col("distancia_mi"),
        F.col("total_pagado"),
    )
    return (
        con_franja.groupBy("fecha", "borough", "franja")
        .agg(
            F.count(F.lit(1)).alias("viajes"),
            F.sum("distancia_mi").alias("distancia"),
            F.sum("total_pagado").alias("ingreso"),
        )
        .select(
            "fecha",
            "borough",
            "franja",
            "viajes",
            F.round("distancia", 2).alias("distancia_total_mi"),
            F.round("ingreso", 2).alias("ingreso_total"),
            F.round(F.col("ingreso") / F.col("distancia"), 2).alias("tarifa_por_milla"),
        )
        .select(*COLUMNAS_TARIFA_POR_MILLA)
    )


# nombre de la tabla en gold -> función que la arma a partir del enriquecido.
TABLAS_GOLD = {
    "tarifa_por_milla_barrio": tarifa_por_milla_por_barrio,
}

# Qué columnas identifican una fila de cada tabla (gold exige que no se repitan ni sean nulas).
LLAVES_DE_GOLD = {
    "tarifa_por_milla_barrio": ("fecha", "borough", "franja"),
}

La probamos con cuatro viajes escritos a mano, de los que sabemos la respuesta:

| Viaje | Barrio | Hora | Millas | Pagó |
|---|---|---|---|---|
| 1 | Manhattan | 10:00 | 2.0 | 20.0 |
| 2 | Manhattan | 11:30 | 3.0 | 15.0 |
| 3 | Queens | 22:15 | 1.0 | 10.0 |
| 4 | (sin barrio) | 03:00 | 4.0 | 8.0 |

Los viajes 1 y 2 caen en la misma franja (mañana) y el mismo barrio: 2 viajes, 5.0 millas, 35.0 pagados, es decir **7.0 por milla**. El viaje 4 no tiene barrio y queda como `Desconocido`, en la madrugada: 8.0 / 4.0 = **2.0 por milla**.

In [ ]:
# Celda 9
# Las horas se escriben como texto y se convierten dentro de Spark: así no dependen de la zona horaria de la máquina.
mano = spark.createDataFrame(
    [
        ("2024-01-05 10:00:00", "Manhattan", 2.0, 20.0),
        ("2024-01-05 11:30:00", "Manhattan", 3.0, 15.0),
        ("2024-01-05 22:15:00", "Queens", 1.0, 10.0),
        ("2024-01-05 03:00:00", None, 4.0, 8.0),
    ],
    "hora string, borough string, distancia_mi double, total_pagado double",
)
mano = mano.select(
    F.to_timestamp("hora").alias("pickup_datetime"),
    F.to_date("hora").alias("pickup_date"),
    "borough",
    "distancia_mi",
    "total_pagado",
)
gold_a_mano = tarifa_por_milla_por_barrio(mano)
gold_a_mano.orderBy("franja").show()

fila = {f["franja"]: f for f in gold_a_mano.collect()}
assert fila["mañana"]["viajes"] == 2 and fila["mañana"]["tarifa_por_milla"] == 7.0
assert fila["madrugada"]["borough"] == "Desconocido" and fila["madrugada"]["tarifa_por_milla"] == 2.0
print("OK: la tarifa por milla de la mañana es 7.0 y la del viaje sin barrio es 2.0.")

## 5. Snowflake: `DDL_TARIFA_POR_MILLA` y `TABLAS_A_CARGAR`

Para cargar gold hay que crear antes la tabla en Snowflake, con **las mismas columnas y tipos compatibles**. El `CREATE TABLE` de la casa lleva `IF NOT EXISTS`, así que se puede ejecutar cuantas veces haga falta, y deja un hueco `{tabla}` que se llena con el nombre de la tabla del entorno (`BIGDATA.DEV.<TABLA>` en dev, `BIGDATA.PROD.<TABLA>` en prod). `TABLAS_A_CARGAR` dice qué tabla de gold va con qué `CREATE TABLE`; para cargar la tuya, agregas una entrada.

Los tipos de Spark se traducen así: `date` es `DATE`, `string` es `VARCHAR`, `bigint` es `NUMBER(38,0)` y `double` es `FLOAT`.

> **Copia a:** `src/pipeline/carga.py`, las constantes `DDL_TARIFA_POR_MILLA` y `TABLAS_A_CARGAR`.

In [ ]:
# Celda 10
# → Copia a src/pipeline/carga.py · DDL_TARIFA_POR_MILLA y TABLAS_A_CARGAR
DDL_TARIFA_POR_MILLA = (
    "CREATE TABLE IF NOT EXISTS {tabla} ("
    "FECHA DATE, "
    "BOROUGH VARCHAR, "
    "FRANJA VARCHAR, "
    "VIAJES NUMBER(38,0), "
    "DISTANCIA_TOTAL_MI FLOAT, "
    "INGRESO_TOTAL FLOAT, "
    "TARIFA_POR_MILLA FLOAT)"
)

# Cada entrada: la tabla de gold y el CREATE TABLE de su tabla en Snowflake.
TABLAS_A_CARGAR = [
    {"gold": "tarifa_por_milla_barrio", "ddl": DDL_TARIFA_POR_MILLA},
]

Dos comprobaciones sin tocar Snowflake: que el `CREATE TABLE` tenga las mismas columnas que gold (en el mismo orden) y cuál sería el nombre completo de la tabla en dev.

In [ ]:
# Celda 11
import re

from pipeline.herramientas import nombre_tabla_snowflake

nombres_del_ddl = re.findall(r"(\w+) (?:DATE|VARCHAR|NUMBER|FLOAT)", DDL_TARIFA_POR_MILLA)
print("Columnas del CREATE TABLE:", [n.lower() for n in nombres_del_ddl])
print("Columnas de gold          :", gold_a_mano.columns)
print("¿Coinciden?               :", [n.lower() for n in nombres_del_ddl] == gold_a_mano.columns)
print()
print("Tabla en dev:", nombre_tabla_snowflake("dev", "tarifa_por_milla_barrio"))
print("Tabla en prod:", nombre_tabla_snowflake("prod", "tarifa_por_milla_barrio"))

## 6. ¿Son las mismas que trae el repositorio?

Estas funciones **ya están** en tus archivos `.py`: llegaron con el `git pull` de la clase 7, exactamente así, con sus dos problemas. Pasar un notebook a un `.py` es copiar la función con su firma; para confirmarlo sin abrir los archivos, la celda de abajo compara el **código** de cada función de este notebook con el del módulo (ignora los comentarios y las explicaciones entre comillas triples) y, para el `CREATE TABLE`, compara el texto.

In [ ]:
# Celda 12
import ast
import inspect

from pipeline import bronze, carga, gold, silver


def codigo_sin_docstring(funcion):
    """El código de la función como árbol, sin su docstring: así solo cuenta lo que ejecuta."""
    arbol = ast.parse(inspect.getsource(funcion).lstrip())
    definicion = arbol.body[0]
    cuerpo = definicion.body
    if cuerpo and isinstance(cuerpo[0], ast.Expr) and isinstance(cuerpo[0].value, ast.Constant):
        definicion.body = cuerpo[1:]
    return ast.dump(definicion)


comparaciones = [
    ("agregar_metadatos", agregar_metadatos, bronze.agregar_metadatos, "bronze.py"),
    ("debe_ingestar", debe_ingestar, bronze.debe_ingestar, "bronze.py"),
    ("registrar_ingesta", registrar_ingesta, bronze.registrar_ingesta, "bronze.py"),
    ("deduplicar", deduplicar, silver.deduplicar, "silver.py"),
    ("_franja_del_dia", _franja_del_dia, gold._franja_del_dia, "gold.py"),
    ("tarifa_por_milla_por_barrio", tarifa_por_milla_por_barrio, gold.tarifa_por_milla_por_barrio, "gold.py"),
]
for nombre, del_notebook, del_modulo, archivo in comparaciones:
    firma_igual = list(inspect.signature(del_notebook).parameters) == list(
        inspect.signature(del_modulo).parameters
    )
    codigo_igual = codigo_sin_docstring(del_notebook) == codigo_sin_docstring(del_modulo)
    print(f"{nombre:<30}{archivo:<11}firma: {'igual' if firma_igual else 'DISTINTA':<10}código: {'igual' if codigo_igual else 'DISTINTO'}")
print(f"{'DDL_TARIFA_POR_MILLA':<30}{'carga.py':<11}texto: {'igual' if DDL_TARIFA_POR_MILLA == carga.DDL_TARIFA_POR_MILLA else 'DISTINTO'}")

Todas deben decir `igual`. Si alguna dice `DISTINTO`, es porque tu `.py` ya tiene cambios tuyos: está bien, y la comparación te dice cuál.

## 7. Qué se copia a cada archivo y qué sigue

Lo que acabas de ver, en una tabla:

| Función | Archivo | Estado |
|---|---|---|
| `agregar_metadatos` | `src/pipeline/bronze.py` | **Pendiente**: la escribe tu equipo (problema 1) |
| `debe_ingestar`, `registrar_ingesta` | `src/pipeline/bronze.py` | Lista |
| `deduplicar` | `src/pipeline/silver.py` | **Pendiente**: la escribe tu equipo (problema 2) |
| `tarifa_por_milla_por_barrio` | `src/pipeline/gold.py` | Lista (ejemplo de la casa; la tuya va junto a esta) |
| `DDL_TARIFA_POR_MILLA`, `TABLAS_A_CARGAR` | `src/pipeline/carga.py` | Lista (la tuya va junto a esta) |

Para comprobar los `.py` desde la terminal se usa `make test`. Hace lo mismo que la celda de abajo: corre las pruebas del equipo y cuenta cada una como `OK`, `PENDIENTE` o `FALLA`. Una prueba sale `PENDIENTE` mientras la función conserve la marca `# PENDIENTE(equipo)`; cuando la escribes y borras la marca, la **misma prueba** pasa a correr de verdad.

**La celda de abajo es la última.** `probar_mis_funciones` recibe las funciones de este notebook, las pone un instante en los módulos del pipeline, corre `tests/test_equipo_bronze.py` y `tests/test_equipo_silver.py`, imprime una línea por prueba y deja todo como estaba. Debes ver `OK` en todo lo que toca la puerta de la tabla de control y `PENDIENTE` en lo de los metadatos y en lo de deduplicar: justo los dos problemas que dijimos en voz alta. Cuando tu equipo escriba esas dos funciones, esas líneas pasarán a `OK`.

In [ ]:
# Celda 13
from pipeline.herramientas import probar_mis_funciones

codigo = probar_mis_funciones(
    agregar_metadatos=agregar_metadatos,
    debe_ingestar=debe_ingestar,
    registrar_ingesta=registrar_ingesta,
    deduplicar=deduplicar,
)
print("Código de salida de pytest:", codigo, "(0 = ninguna prueba falló)")